# Simple RAG Exercise

Build a simple RAG flow to recommend oldie movies based on user's requests. The dataset includes 5,000 movies descriptions. In the exercise, you will learn to add a filter to the semantic retrieval and the data columns sent to the generation step.

Fill in the empty cells, and answer the questions on the course site.

In [1]:
from rich.console import Console

console = Console()

## Loading the Movie Dataset

We will load the moview dataset from Hugging Face hub in:
https://huggingface.co/datasets/AiresPucrs/tmdb-5000-movies

In [2]:
from datasets import load_dataset

dataset = load_dataset(
    "AiresPucrs/tmdb-5000-movies",
    split="train"
)

In [3]:
console.print(dataset)

Dataset({
    features: ['id', 'budget', 'genres', 'homepage', 'keywords', 'original_language', 'original_title', 'overview',
'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 
'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count', 'cast', 'crew'],
    num_rows: 4803
})

## Encode using Vector Embedding

We will use one of the popular open source vector databases, [Qdrant](https://qdrant.tech/), and one of the popular embedding encoder and text transformer libraries, [SentenceTransformer](https://sbert.net/).

This time we will use the following sentence similarity model:
https://huggingface.co/sentence-transformers/all-mpnet-base-v2

In [4]:
from qdrant_client import models, QdrantClient
from sentence_transformers import SentenceTransformer

# create the vector database client
qdrant = QdrantClient(":memory:")

# Create the embedding encoder
encoder = SentenceTransformer(
    "sentence-transformers/all-mpnet-base-v2"
)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [5]:
console.print(encoder)

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 
'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 
'MPNetModel'})
  (1): Pooling({'embedding_dimension': 768, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)

In [6]:
# Create collection to store the wine rating data
collection_name="movies"

qdrant.recreate_collection(
    collection_name=collection_name,
    vectors_config=models.VectorParams(
        size=encoder.get_sentence_embedding_dimension(), # Vector size is defined by used model
        distance=models.Distance.COSINE
    )
)

C:\Users\2025\AppData\Local\Temp\ipykernel_13556\2094978642.py:7: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  size=encoder.get_sentence_embedding_dimension(), # Vector size is defined by used model
C:\Users\2025\AppData\Local\Temp\ipykernel_13556\2094978642.py:4: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  qdrant.recreate_collection(


True

### Loading the data into the vector database

We will use the collection that we created above, to go over all the rows and encode the `overview` column of the wine dataset, encode it with the encoder into embedding vector, and store it in the vector database. Please use the index of the movie from the dataset (`id` column) as the `id` in the vector index.

Please note that some of the rows are missing the `overview`. You should ignore them and not upload them into the vector database index.

This step will take a few seconds (less than a minute on my laptop).

In [7]:

#-------------------------------------------------------------------------------------
from rich.progress import track

# 1. استخراج النصوص والمعرفات والقواميس في قوائم منفصلة
movies_with_overview = [m for m in dataset if m["overview"]]
documents = [m["overview"] for m in movies_with_overview]
ids = [m["id"] for m in movies_with_overview]
payloads = [dict(m) for m in movies_with_overview]

# 2. توليد الـ Embeddings لجميع الأفلام دفعة واحدة (أسرع بمئات المرات!)
print("جاري تحويل النصوص إلى مستندات رقمية (Embeddings)...")
vectors = encoder.encode(documents, batch_size=64, show_progress_bar=True).tolist()

# 3. رفع البيانات إلى Qdrant على دفعات وبشريط تقدم ذكي
print("جاري رفع البيانات إلى قاعدة متجهات Qdrant...")
points = [
    models.PointStruct(id=ids[i], vector=vectors[i], payload=payloads[i])
    for i in range(len(movies_with_overview))
]

# رفع البيانات على دفعات حجمها 100 لضمان السلاسة
batch_size = 100
for i in track(range(0, len(points), batch_size), description="جاري الرفع..."):
    qdrant.upload_points(
        collection_name=collection_name,
        points=points[i : i + batch_size]
    )

print("تمت العملية بنجاح وبسرعة فائقة! 🎉")

جاري تحويل النصوص إلى مستندات رقمية (Embeddings)...


Batches:   0%|          | 0/75 [00:00<?, ?it/s]

جاري رفع البيانات إلى قاعدة متجهات Qdrant...


Output()

تمت العملية بنجاح وبسرعة فائقة! 🎉


In [8]:
console.print(
    qdrant
    .get_collection(
        collection_name=collection_name
    )
)

CollectionInfo(
    status=<CollectionStatus.GREEN: 'green'>,
    optimizer_status=<OptimizersStatusOneOf.OK: 'ok'>,
    warnings=None,
    indexed_vectors_count=0,
    points_count=4800,
    segments_count=1,
    config=CollectionConfig(
        params=CollectionParams(
            vectors=VectorParams(
                size=768,
                distance=<Distance.COSINE: 'Cosine'>,
                hnsw_config=None,
                quantization_config=None,
                on_disk=None,
                datatype=None,
                multivector_config=None
            ),
            shard_number=None,
            sharding_method=None,
            replication_factor=None,
            write_consistency_factor=None,
            read_fan_out_factor=None,
            read_fan_out_delay_ms=None,
            on_disk_payload=None,
            sparse_vectors=None
        ),
        hnsw_config=HnswConfig(
            m=16,
            ef_construct=100,
            full_scan_threshold=10000,
            max_indexing_threads=0,
            on_disk=None,
            payload_m=None,
            inline_storage=None
        ),
        optimizer_config=OptimizersConfig(
            deleted_threshold=0.2,
            vacuum_min_vector_number=1000,
            default_segment_number=0,
            max_segment_size=None,
            memmap_threshold=None,
            indexing_threshold=20000,
            flush_interval_sec=5,
            max_optimization_threads=1,
            prevent_unoptimized=None
        ),
        wal_config=WalConfig(wal_capacity_mb=32, wal_segments_ahead=0, wal_retain_closed=1),
        quantization_config=None,
        strict_mode_config=None,
        metadata=None
    ),
    payload_schema={},
    update_queue=None
)

## **R**etrieve sematically relevant data based on user's query

Once the data is loaded into the vector database and the indexing process is done, we can start using our simple RAG system.

In [9]:
user_prompt = "Love story between an Asian king and European teacher"

### Encoding the user's query

We will use the same encoder that we used to encode the document data to encode the query of the user. 
This way we can search results based on semantic similarity. 

In [10]:
query_vector = encoder.encode(user_prompt).tolist()

### Create filter on the results

We only want movies from the '90s. Please create a filter base on the `release_date` column. Check the Qdrant documentation in: https://qdrant.tech/documentation/concepts/filtering/#datetime-range

In [11]:
from qdrant_client import models

query_filter = models.Filter(
    must=[
        models.FieldCondition(
            key="release_date",
            range=models.DatetimeRange(
                gte="1990-01-01T00:00:00Z",
                lt="2000-01-01T00:00:00Z"
            )
        )
    ]
)

### Search similar rows

We can now take the embedding encoding of the user's query and use it to find similar rows in the vector database.

In [12]:

# ------------------------------------------------------------------
# الكود المحدث المتوافق مع النسخة الجديدة (تم تعديل اسم المعامل إلى query)
hits = qdrant.query_points(
    collection_name=collection_name,
    query=query_vector,          # التعديل هنا: غيّرنا الاسم ليكون query فقط
    limit=5,
    query_filter=query_filter
).points

In [13]:
from rich.text import Text
from rich.table import Table

table = Table(title="Retrieval Results", show_lines=True)

table.add_column("ID", style="bright_red")
table.add_column("Original Title", style="bright_red")
table.add_column("Overview", style="bright_red")
table.add_column("Score", style="bright_red")

for hit in hits:
    table.add_row(
        str(hit.payload["id"]),
        hit.payload["original_title"],
        f'{hit.payload["overview"]}',
        f"{hit.score:.4f}"
    )

console.print(table)

                                                 Retrieval Results                                                 
┏━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ ID    ┃ Original Title                                 ┃ Overview                                      ┃ Score  ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ 1439  │ Anna and the King                              │ The story of the romance between the King of  │ 0.6419 │
│       │                                                │ Siam (now Thailand) and the widowed British   │        │
│       │                                                │ school teacher Anna Leonowens during the      │        │
│       │                                                │ 1860's. Anna teaches the children and becomes │        │
│       │                                                │ romanced by the King. She convinces him that  │        │
│       │                                                │ a man can be loved by just one woman.         │        │
├───────┼────────────────────────────────────────────────┼───────────────────────────────────────────────┼────────┤
│ 29371 │ The Incredibly True Adventure of Two Girls In  │ An adventurous love story between two young   │ 0.4329 │
│       │ Love                                           │ women of different social and economic        │        │
│       │                                                │ backgrounds who find themselves going through │        │
│       │                                                │ all the typical struggles of a new romance.   │        │
├───────┼────────────────────────────────────────────────┼───────────────────────────────────────────────┼────────┤
│ 41469 │ Next Stop Wonderland                           │ A lighthearted story about a man and a woman  │ 0.4282 │
│       │                                                │ who seem destined to be together... and the   │        │
│       │                                                │ hilarious chain of accidents that seem        │        │
│       │                                                │ determined to keep them apart!                │        │
├───────┼────────────────────────────────────────────────┼───────────────────────────────────────────────┼────────┤
│ 666   │ Central do Brasil                              │ An emotive journey of a former school         │ 0.4241 │
│       │                                                │ teacher, who writes letters for illiterate    │        │
│       │                                                │ people, and a young boy, whose mother has     │        │
│       │                                                │ just died, as they search for the father he   │        │
│       │                                                │ never knew.                                   │        │
├───────┼────────────────────────────────────────────────┼───────────────────────────────────────────────┼────────┤
│ 8587  │ The Lion King                                  │ A young lion cub named Simba can't wait to be │ 0.4204 │
│       │                                                │ king. But his uncle craves the title for      │        │
│       │                                                │ himself and will stop at nothing to get it.   │        │
└───────┴────────────────────────────────────────────────┴───────────────────────────────────────────────┴────────┘

## **A**ugment the prompt to the LLM with retrieved data

In our simple example, we will simply take the top result and use it in the prompt to the generation LLM. We will filter some of the columns and keep only the following:
* `original_title`
* `title`
* `overview`
* `release_date`
* `popularity`

In [14]:
# define a variable to hold the search results with specific fields
search_results = [
    {
        "original_title": hit.payload["original_title"],
        "title": hit.payload["title"],
        "overview": hit.payload["overview"],
        "release_date": hit.payload["release_date"],
        "popularity": hit.payload["popularity"]
    }
    for hit in hits
]

In [15]:
console.print(search_results)

[
    {
        'original_title': 'Anna and the King',
        'title': 'Anna and the King',
        'overview': "The story of the romance between the King of Siam (now Thailand) and the widowed British 
school teacher Anna Leonowens during the 1860's. Anna teaches the children and becomes romanced by the King. She 
convinces him that a man can be loved by just one woman.",
        'release_date': '1999-12-16',
        'popularity': 13.327372
    },
    {
        'original_title': 'The Incredibly True Adventure of Two Girls In Love',
        'title': 'The Incredibly True Adventure of Two Girls In Love',
        'overview': 'An adventurous love story between two young women of different social and economic backgrounds
who find themselves going through all the typical struggles of a new romance.',
        'release_date': '1995-06-16',
        'popularity': 1.70819
    },
    {
        'original_title': 'Next Stop Wonderland',
        'title': 'Next Stop Wonderland',
        'overview': 'A lighthearted story about a man and a woman who seem destined to be together... and the 
hilarious chain of accidents that seem determined to keep them apart!',
        'release_date': '1998-08-21',
        'popularity': 1.334481
    },
    {
        'original_title': 'Central do Brasil',
        'title': 'Central Station',
        'overview': 'An emotive journey of a former school teacher, who writes letters for illiterate people, and a
young boy, whose mother has just died, as they search for the father he never knew.',
        'release_date': '1998-01-16',
        'popularity': 5.928937
    },
    {
        'original_title': 'The Lion King',
        'title': 'The Lion King',
        'overview': "A young lion cub named Simba can't wait to be king. But his uncle craves the title for himself
and will stop at nothing to get it.",
        'release_date': '1994-06-23',
        'popularity': 90.457886
    }
]

## **G**enerate reply to the user's query

We will use GPT-4 from [OpenAI](https://platform.openai.com/docs/models). Please write the prompt to instruct the LLM to write the recommendations based on the search results.

In [16]:

from openai import OpenAI

# 1. تعريف الاتصال بالسيرفر المحلي
client = OpenAI(
    base_url = 'http://localhost:11434/v1',
    api_key='ollama'
)

# 2. استخراج بيانات الفيلم
movie_title = hits[0].payload['title']
movie_overview = hits[0].payload['overview']

# 3. صياغة الـ Prompt بالإنجليزية للأستاذ
prompt = f"""
You are a professional movie recommendation assistant. 
Based on the following retrieved movie context, explain to the user in a clear and engaging English response why this movie is a great recommendation for them.

Movie Title: {movie_title}
Movie Description: {movie_overview}
"""

# 4. استدعاء الموديل
response = client.chat.completions.create(
    model="llama3", 
    messages=[{"role": "user", "content": prompt}]
)

# 5. طباعة النتيجة النهائية
print(f"--- FINAL SYSTEM RECOMMENDATION ---\n")
print(response.choices[0].message.content)

--- FINAL SYSTEM RECOMMENDATION ---

What a fantastic discovery!

I think "Anna and the King" would be an excellent recommendation for you, and here's why:

**Timeless romance**: This beautiful period drama tells the story of an unlikely love affair between two people from different worlds. The King of Siam and Anna Leonowens face cultural differences, societal expectations, and personal doubts as they navigate their feelings for each other. Their romance is a testament to the power of true love, making it a timeless and universally relatable tale.

**Unique setting**: Thailand in the 1860s provides a fascinating backdrop for this romance. The film showcases the vibrant culture, stunning architecture, and rich traditions of the Kingdom of Siam (now Thailand). This unique setting adds depth and flavor to the story, offering a visually breathtaking experience.

**Strong leading performances**: Jodie Foster gives a captivating performance as Anna Leonowens, bringing warmth and intelligenc

In [17]:
!pip install openai
